# 학생 평가기 — Team 9

- 별도 `pip install` 없이 Gemini REST API 사용
- 골드셋의 `id`, `question`, `gold_articles`, `key_facts`를 동적으로 처리
- MRR 20점 + KeyFact F1 30점 + Gemini 판정 50점
- 최종 제출 파일: `eval_9.json`
- 출력 형식: `results -> blind_id, total, status`

코드는 너무 세로로 늘어지지 않도록 **함수/변수 묶음 사이에만 빈 줄을 두고**, 일반 문장은 가능한 한 한 줄에 작성했습니다.


In [23]:
# 1. 설정
import json, os, re, time, unicodedata, urllib.error, urllib.request
from collections import Counter
from pathlib import Path

TEAM_ID = "9"
MODEL_ID = "gemini-3.5-flash"
API_BASE = "https://generativelanguage.googleapis.com/v1beta/models"

WEIGHT_MRR, WEIGHT_KEYFACT, WEIGHT_JUDGE = 20.0, 30.0, 50.0
AXIS_WEIGHTS = {"accuracy": 0.40, "grounding": 0.25, "completeness": 0.20, "clarity": 0.15}

RETRIEVAL_CUTOFF = 4
MAX_RETRY, RETRY_BACKOFF, REQUEST_INTERVAL, TIMEOUT_S = 3, 4.0, 0.6, 120

OUTPUT_PATH = Path(f"/content/eval_{TEAM_ID}.json")
DETAIL_PATH = Path(f"/content/eval_detail_{TEAM_ID}.json")

print("설정 완료:", MODEL_ID)


설정 완료: gemini-3.5-flash


In [24]:
# 2. API 키 + 파일 업로드
def load_api_key():
    key = os.environ.get("GEMINI_API_KEY")
    if key:
        print("[키] 환경변수에서 읽었습니다.")
        return key.strip()

    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key:
            print("[키] Colab Secret에서 읽었습니다.")
            return key.strip()
    except Exception:
        pass

    import getpass
    key = getpass.getpass("GEMINI_API_KEY를 입력하세요(화면에 표시되지 않습니다): ").strip()
    if not key:
        raise RuntimeError("GEMINI_API_KEY가 비어 있습니다.")
    return key


try:
    from google.colab import files
    uploaded = files.upload()
    uploaded_paths = [Path("/content") / name for name in uploaded]
except Exception:
    uploaded_paths = list(Path(".").glob("*.json"))


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


loaded = []
for path in uploaded_paths:
    try:
        loaded.append((path, read_json(path)))
    except Exception as e:
        print(f"[건너뜀] {path.name}: {type(e).__name__}: {e}")

gold_candidates = [(p, d) for p, d in loaded if isinstance(d, dict) and isinstance(d.get("questions"), list)]
answer_candidates = [(p, d) for p, d in loaded if isinstance(d, dict) and isinstance(d.get("answers"), list)]

if len(gold_candidates) != 1:
    raise ValueError(f"골드셋 JSON은 정확히 1개여야 합니다. 현재 {len(gold_candidates)}개")
if not answer_candidates:
    raise ValueError("답변 JSON을 찾지 못했습니다.")

GOLD_PATH, GOLD_RAW = gold_candidates[0]
ANSWER_FILES = answer_candidates

print("골드셋:", GOLD_PATH.name, "/ 문항:", len(GOLD_RAW["questions"]))
print("답변 파일:", [p.name for p, _ in ANSWER_FILES])


Saving answers_public_4.json to answers_public_4 (1).json
Saving answers_public_7.json to answers_public_7 (1).json
Saving answers_public_13.json to answers_public_13 (1).json
Saving answers_public_14.json to answers_public_14 (1).json
Saving answers_public_15.json to answers_public_15 (2).json
Saving gold_questions_public10.json to gold_questions_public10 (2).json
골드셋: gold_questions_public10 (2).json / 문항: 10
답변 파일: ['answers_public_4 (1).json', 'answers_public_7 (1).json', 'answers_public_13 (1).json', 'answers_public_14 (1).json', 'answers_public_15 (2).json']


In [25]:
# 3. MRR + KeyFact F1 계산
_TOKEN_RE = re.compile(r"[0-9]+|[A-Za-z]+|[가-힣]+")
_JOSA = ("으로써", "에게서", "이라고", "에서는", "으로는", "입니다", "습니다", "합니다", "에서", "에게", "으로",
         "라고", "까지", "부터", "보다", "처럼", "이나", "마다", "조차", "한테", "와의", "과의",
         "의", "가", "이", "은", "는", "을", "를", "와", "과", "로", "도", "만", "에")
_STOP = {"및", "등", "그", "저", "것", "수", "때", "경우", "대한", "대하여", "관한", "위하여", "위한",
         "있는", "하는", "그리고", "또는", "또한", "따라", "따른"}


def norm_doc(text):
    return re.sub(r"\s+", "", unicodedata.normalize("NFC", str(text)))


def art_no(value):
    m = re.search(r"(\d+)", str(value))
    return int(m.group(1)) if m else None


def content_tokens(text):
    result = []

    for word in _TOKEN_RE.findall(unicodedata.normalize("NFC", str(text))):
        word = word.lower()

        if len(word) < 2 or word in _STOP:
            continue

        result.append(word)

    return result


def keyfact_f1(answer, key_facts):
    ref = Counter(token for fact in key_facts for token in content_tokens(fact))
    hyp = Counter(content_tokens(answer))
    if not ref or not hyp:
        return 0.0, 0.0, 0.0

    overlap = sum((ref & hyp).values())
    precision, recall = overlap / sum(hyp.values()), overlap / sum(ref.values())
    f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
    return f1, precision, recall


def parse_retrieved_item(item):
    if isinstance(item, dict):
        doc = item.get("doc") or item.get("document") or item.get("doc_name")
        article = item.get("article") if item.get("article") is not None else item.get("article_no")
        return (norm_doc(doc), art_no(article)) if doc is not None and article is not None else None
    if isinstance(item, (list, tuple)) and len(item) >= 2:
        return norm_doc(item[0]), art_no(item[1])
    return None


def retrieval_rr(retrieved, gold_articles, cutoff=RETRIEVAL_CUTOFF):
    gold = {(norm_doc(g["doc"]), art_no(g["article"])) for g in gold_articles
            if isinstance(g, dict) and g.get("doc") is not None and g.get("article") is not None}

    if not isinstance(retrieved, (list, tuple)):
        return 0.0

    for rank, item in enumerate(retrieved[:cutoff], start=1):
        if parse_retrieved_item(item) in gold:
            return 1.0 / rank
    return 0.0


In [26]:
# 4. 골드셋 / 후보 ID 처리
def load_gold(raw):
    gold, duplicate_ids = {}, []

    for q in raw.get("questions", []):
        if q.get("id") is None:
            raise ValueError("골드셋 문항에 id가 없는 항목이 있습니다.")

        qid = str(q["id"])
        if qid in gold:
            duplicate_ids.append(qid)

        gold[qid] = {
            "question": q.get("question", ""),
            "gold_articles": q.get("gold_articles", []),
            "key_facts": q.get("key_facts", [])
        }

    if duplicate_ids:
        raise ValueError(f"골드셋 id 중복: {sorted(set(duplicate_ids))}")
    return gold


def find_blind_id(path, data):
    for key in ("blind_id", "blindId"):
        value = data.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip().upper()

    meta = data.get("meta")
    if isinstance(meta, dict):
        value = meta.get("blind_id") or meta.get("blindId")
        if isinstance(value, str) and value.strip():
            return value.strip().upper()

    m = re.search(r"BLIND\s*0*(\d+)", unicodedata.normalize("NFC", Path(path).name), re.I)
    if m:
        return f"BLIND{int(m.group(1)):02d}"

    if data.get("team") is not None and str(data["team"]).strip():
        return str(data["team"]).strip()

    return Path(path).stem


GOLD = load_gold(GOLD_RAW)
print("문항 ID:", list(GOLD))


문항 ID: ['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10']


In [27]:
# 5. Gemini 판정 기준
JUDGE_SYSTEM = """당신은 약관 질의응답 답변을 채점하는 엄격한 심사자입니다.
오직 제공된 질문, 정답 근거 조항, 정답 키팩트, 참가자 답변만 사용하십시오.
외부 지식으로 정답을 보충하거나 참가자에게 유리하게 추정하지 마십시오.

네 축을 각각 0~10 정수로 채점합니다.
- accuracy: 답변의 사실이 정답과 일치하는가. 왜곡, 환각, 잘못된 숫자·주체·조건이 있으면 낮춥니다.
- grounding: 답변의 사실과 답변에서 명시한 약관명·조항 번호가 정답 근거 조항과 일치하는지 평가합니다.
  답변이 특정 약관명이나 조항 번호를 명시했는데 정답 gold_articles와 다르면 grounding을 감점하십시오.
  잘못된 문서와 조항을 정답 근거처럼 제시한 경우 크게 감점하십시오.
  단, 답변에 약관명이나 조항 번호를 아예 적지 않은 경우에는 그것만으로 감점하지 마십시오.
- completeness: 질문이 요구하는 핵심 사실과 키팩트를 빠짐없이 담았는가. 일부만 답하면 낮춥니다.
- clarity: 질문에 직접 답하고 군더더기 없이 이해하기 쉬운가.

짧다는 이유만으로 감점하지 말고 의미가 같으면 같은 사실로 인정하십시오.
답변이 비어 있거나 질문과 무관하면 각 축을 0~2점으로 평가하십시오.
각 이유는 한 문장으로 짧게 작성하고 반드시 지정된 JSON 형식으로만 출력하십시오."""

JUDGE_RESPONSE_SCHEMA = {
    "type": "OBJECT",
    "properties": {
        "accuracy": {"type": "INTEGER", "minimum": 0, "maximum": 10},
        "accuracy_reason": {"type": "STRING"},
        "grounding": {"type": "INTEGER", "minimum": 0, "maximum": 10},
        "grounding_reason": {"type": "STRING"},
        "completeness": {"type": "INTEGER", "minimum": 0, "maximum": 10},
        "completeness_reason": {"type": "STRING"},
        "clarity": {"type": "INTEGER", "minimum": 0, "maximum": 10},
        "clarity_reason": {"type": "STRING"}
    },
    "required": ["accuracy", "accuracy_reason", "grounding", "grounding_reason",
                 "completeness", "completeness_reason", "clarity", "clarity_reason"]
}


def build_judge_prompt(question, gold_articles, key_facts, answer, retrieved):
    citations = [str(g.get("citation") or f'{g.get("doc", "")} 제{g.get("article", "")}조')
                 for g in gold_articles if isinstance(g, dict)]
    cites_text = "\n".join(f"- {c}" for c in citations) or "(없음)"
    facts_text = "\n".join(f"{i}. {fact}" for i, fact in enumerate(key_facts, start=1)) or "(없음)"

    picked = []
    for item in (retrieved or [])[:RETRIEVAL_CUTOFF]:
        if isinstance(item, (list, tuple)) and len(item) >= 2:
            picked.append(f"{item[0]} 제{item[1]}조")
        elif isinstance(item, dict):
            doc = item.get("doc") or item.get("document") or item.get("doc_name")
            article = item.get("article", item.get("article_no"))
            if doc is not None and article is not None:
                picked.append(f"{doc} 제{article}조")

    picked_text = ", ".join(picked) or "(없음)"
    return (f"[질문]\n{question}\n\n[정답 근거 조항]\n{cites_text}\n\n[정답 키팩트]\n{facts_text}\n\n"
            f"[참가자가 검색한 근거]\n{picked_text}\n\n[참가자 답변]\n{answer or '(빈 답변)'}\n\n"
            "위 기준에 따라 네 축을 채점하십시오.")


def judge_score(axes):
    total = 0.0
    for name, weight in AXIS_WEIGHTS.items():
        value = axes.get(name, 0)
        value = max(0.0, min(10.0, float(value))) if isinstance(value, (int, float)) else 0.0
        total += value * weight
    return total * 10.0


In [28]:
# 6. Gemini REST API 호출
def _post_json(url, payload, api_key):
    request = urllib.request.Request(
        url, data=json.dumps(payload, ensure_ascii=False).encode("utf-8"),
        headers={"Content-Type": "application/json", "x-goog-api-key": api_key}, method="POST"
    )
    with urllib.request.urlopen(request, timeout=TIMEOUT_S) as response:
        return json.loads(response.read().decode("utf-8"))


def _extract_response_text(data):
    candidates = data.get("candidates") or []
    if not candidates:
        raise ValueError("Gemini candidates가 없습니다.")

    texts = [part["text"] for part in candidates[0].get("content", {}).get("parts", [])
             if isinstance(part, dict) and part.get("thought") is not True and isinstance(part.get("text"), str)]

    if not texts:
        raise ValueError("Gemini 최종 text 응답이 없습니다.")
    return "".join(texts).strip()


def validate_axes(obj):
    if not isinstance(obj, dict):
        raise ValueError("Gemini JSON 응답이 객체가 아닙니다.")

    for name in ("accuracy", "grounding", "completeness", "clarity"):
        if name not in obj or not isinstance(obj[name], (int, float)) or not 0 <= float(obj[name]) <= 10:
            raise ValueError(f"{name} 점수 오류: {obj.get(name)}")
    return obj


def call_judge(prompt, api_key, usage):
    url = f"{API_BASE}/{MODEL_ID}:generateContent"

    base_generation = {
        "responseMimeType": "application/json",
        "responseSchema": JUDGE_RESPONSE_SCHEMA,
        "maxOutputTokens": 512
    }
    contents = [{"role": "user", "parts": [{"text": prompt}]}]
    system_instruction = {"parts": [{"text": JUDGE_SYSTEM}]}

    payloads = [
        {"systemInstruction": system_instruction, "contents": contents,
         "generationConfig": {**base_generation, "thinkingConfig": {"thinkingLevel": "low"}}},
        {"systemInstruction": system_instruction, "contents": contents, "generationConfig": base_generation}
    ]

    last_error = None

    for variant, payload in enumerate(payloads, start=1):
        for attempt in range(MAX_RETRY):
            try:
                data = _post_json(url, payload, api_key)
                meta = data.get("usageMetadata", {})

                usage["in"] += int(meta.get("promptTokenCount", 0) or 0)
                usage["out"] += int(meta.get("candidatesTokenCount", 0) or 0)
                usage["thoughts"] += int(meta.get("thoughtsTokenCount", 0) or 0)
                usage["calls"] += 1

                text = _extract_response_text(data)
                cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.I).strip()
                return validate_axes(json.loads(cleaned))

            except urllib.error.HTTPError as e:
                body = e.read().decode("utf-8", "replace")[:500]
                last_error = f"HTTP {e.code}: {body}"

                if e.code == 400:
                    print(f"      [HTTP 400] payload 변형 {variant} 실패")
                    break

                if attempt + 1 < MAX_RETRY:
                    wait = RETRY_BACKOFF * (2 ** attempt)
                    print(f"      [재시도] {wait:.1f}초 후")
                    time.sleep(wait)

            except Exception as e:
                last_error = f"{type(e).__name__}: {e}"
                if attempt + 1 < MAX_RETRY:
                    wait = RETRY_BACKOFF * (2 ** attempt)
                    print(f"      [재시도] {wait:.1f}초 후")
                    time.sleep(wait)

    print(f"      [판정 실패] {last_error}")
    return None


In [29]:
# 7. 후보 한 개 평가
def evaluate_candidate(blind_id, data, gold, api_key, usage):
    raw_answers = data.get("answers", [])
    qid_counts = Counter(str(a["qid"]) for a in raw_answers if isinstance(a, dict) and a.get("qid") is not None)
    duplicate_qids = sorted(qid for qid, count in qid_counts.items() if count > 1)

    answers = {}
    for answer in raw_answers:
        if isinstance(answer, dict) and answer.get("qid") is not None:
            answers.setdefault(str(answer["qid"]), answer)

    gold_qids, answer_qids = set(gold), set(answers)
    missing_qids = sorted(gold_qids - answer_qids)
    unknown_qids = sorted(answer_qids - gold_qids)
    structural_problem = bool(duplicate_qids or missing_qids or unknown_qids)

    if duplicate_qids:
        print("   [구조] 중복 qid:", duplicate_qids)
    if missing_qids:
        print("   [구조] 누락 qid:", missing_qids)
    if unknown_qids:
        print("   [구조] 알 수 없는 qid:", unknown_qids)

    per_question, n_judge_fail = [], 0

    for index, (qid, g) in enumerate(gold.items(), start=1):
        a = answers.get(qid)

        if a is None:
            per_question.append({"qid": qid, "rr": 0.0, "keyfact_f1": 0.0, "keyfact_precision": 0.0,
                                 "keyfact_recall": 0.0, "judge": 0.0, "axes": None, "error": "missing_qid"})
            print(f"   !! [{index}/{len(gold)}] {qid} 누락 → 0점")
            continue

        answer_text, retrieved = a.get("answer", "") or "", a.get("retrieved", []) or []
        rr = retrieval_rr(retrieved, g["gold_articles"])
        f1, precision, recall = keyfact_f1(answer_text, g["key_facts"])

        if answer_text.strip():
            prompt = build_judge_prompt(g["question"], g["gold_articles"], g["key_facts"], answer_text, retrieved)
            axes = call_judge(prompt, api_key, usage)
            time.sleep(REQUEST_INTERVAL)
        else:
            axes = {"accuracy": 0, "grounding": 0, "completeness": 0, "clarity": 0,
                    "accuracy_reason": "빈 답변", "grounding_reason": "빈 답변",
                    "completeness_reason": "빈 답변", "clarity_reason": "빈 답변"}

        judge = None if axes is None else judge_score(axes)
        if judge is None:
            n_judge_fail += 1

        per_question.append({"qid": qid, "rr": rr, "keyfact_f1": f1, "keyfact_precision": precision,
                             "keyfact_recall": recall, "judge": judge, "axes": axes})

        jtxt = f"{judge:5.1f}" if judge is not None else "---"
        print(f"   [{index}/{len(gold)}] {qid}  RR {rr:.2f}  F1 {f1:.4f}  판정 {jtxt}")

    if not per_question:
        return {"blind_id": blind_id, "total": None, "status": "failed"}, per_question

    judge_ok = [row for row in per_question if row["judge"] is not None]
    if not judge_ok:
        return {"blind_id": blind_id, "total": None, "status": "failed"}, per_question

    n = len(per_question)
    mrr = sum(row["rr"] for row in per_question) / n
    keyfact = sum(row["keyfact_f1"] for row in per_question) / n
    judge_avg = sum(row["judge"] for row in judge_ok) / len(judge_ok)

    total = WEIGHT_MRR * mrr + WEIGHT_KEYFACT * keyfact + WEIGHT_JUDGE * (judge_avg / 100.0)
    total = max(0.0, min(100.0, total))
    status = "partial" if n_judge_fail > 0 or structural_problem else "completed"

    print(f"   → MRR {mrr:.4f} / KeyFact {keyfact:.4f} / Judge {judge_avg:.2f} / Total {total}")
    return {"blind_id": blind_id, "total": total, "status": status}, per_question


In [30]:
# 8. 전체 평가 + eval_9.json 저장
api_key = load_api_key()
usage = {"in": 0, "out": 0, "thoughts": 0, "calls": 0}

results, detail = [], {}
candidate_ids = [find_blind_id(path, data) for path, data in ANSWER_FILES]
duplicate_candidate_ids = {cid for cid, count in Counter(candidate_ids).items() if count > 1}

print(f"\n[골드] 문항 {len(GOLD)}개")
print(f"[후보] {len(ANSWER_FILES)}개")
print(f"[예상 Gemini 호출] {len(ANSWER_FILES) * len(GOLD)}회")

for path, data in ANSWER_FILES:
    candidate_id = find_blind_id(path, data)
    print("\n" + "=" * 70)
    print(candidate_id)
    print("=" * 70)

    row, per_q = evaluate_candidate(candidate_id, data, GOLD, api_key, usage)

    if candidate_id in duplicate_candidate_ids and row["status"] == "completed":
        row["status"] = "partial"

    results.append(row)
    detail[candidate_id] = per_q

final_output = {"results": results}
OUTPUT_PATH.write_text(json.dumps(final_output, ensure_ascii=False, indent=2), encoding="utf-8")
DETAIL_PATH.write_text(json.dumps(detail, ensure_ascii=False, indent=2), encoding="utf-8")

print("\n평가 결과")
for row in results:
    print(row)

print(f"\nAPI 성공 호출: {usage['calls']}회")
print(f"입력 {usage['in']:,} / 출력 {usage['out']:,} / thinking {usage['thoughts']:,} tokens")
print("제출 파일:", OUTPUT_PATH)


[키] Colab Secret에서 읽었습니다.

[골드] 문항 10개
[후보] 5개
[예상 Gemini 호출] 50회

4
   [1/10] P01  RR 1.00  F1 0.8750  판정 100.0
   [2/10] P02  RR 1.00  F1 0.3019  판정 100.0
   [3/10] P03  RR 1.00  F1 0.7810  판정 100.0
   [4/10] P04  RR 1.00  F1 0.6857  판정  90.0
   [5/10] P05  RR 1.00  F1 0.7083  판정 100.0
   [6/10] P06  RR 1.00  F1 0.4842  판정 100.0
   [7/10] P07  RR 1.00  F1 0.4583  판정  98.0
   [8/10] P08  RR 1.00  F1 0.4255  판정  20.0
   [9/10] P09  RR 1.00  F1 0.2000  판정  96.0
   [10/10] P10  RR 1.00  F1 0.5610  판정 100.0
   → MRR 1.0000 / KeyFact 0.5481 / Judge 90.40 / Total 81.64281453025501

7
   [1/10] P01  RR 1.00  F1 0.8333  판정 100.0
   [2/10] P02  RR 1.00  F1 0.3774  판정 100.0
   [3/10] P03  RR 1.00  F1 0.7788  판정 100.0
   [4/10] P04  RR 1.00  F1 0.7805  판정 100.0
   [5/10] P05  RR 1.00  F1 0.5797  판정 100.0
   [6/10] P06  RR 1.00  F1 0.5974  판정 100.0
   [7/10] P07  RR 1.00  F1 0.6061  판정  98.0
   [8/10] P08  RR 1.00  F1 0.3333  판정 100.0
   [9/10] P09  RR 1.00  F1 0.5556  판정  94.0
   [10/10] P10  RR

In [31]:
# 9. 출력 형식 검증 + 다운로드
check = json.loads(OUTPUT_PATH.read_text(encoding="utf-8"))

assert set(check) == {"results"}
for row in check["results"]:
    assert set(row) == {"blind_id", "total", "status"}
    assert row["status"] in {"completed", "partial", "failed"}

    if row["status"] == "failed":
        assert row["total"] is None
    if row["total"] is not None:
        assert 0.0 <= float(row["total"]) <= 100.0

print("JSON 형식 검증 PASS")
print(json.dumps(check, ensure_ascii=False, indent=2))

try:
    from google.colab import files
    files.download(str(OUTPUT_PATH))
except Exception:
    pass


JSON 형식 검증 PASS
{
  "results": [
    {
      "blind_id": "4",
      "total": 81.64281453025501,
      "status": "completed"
    },
    {
      "blind_id": "7",
      "total": 87.64293072637628,
      "status": "completed"
    },
    {
      "blind_id": "13",
      "total": 88.68423052473747,
      "status": "completed"
    },
    {
      "blind_id": "14",
      "total": 89.02333612813575,
      "status": "completed"
    },
    {
      "blind_id": "15",
      "total": 85.34214812978979,
      "status": "completed"
    }
  ]
}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>